# 7. Analysis: trade studies, sizing, and requirement consistency

`sysml2.analysis` gives executable models analytical teeth by projecting
them onto external solvers. Three submodules, each behind its own extra
(the core package stays dependency-light):

| Module | Solver | Install | Answers |
|---|---|---|---|
| `analysis.trades` | OR-Tools CP-SAT | `pip install "longeron[trades]"` | which discrete component mixes are feasible / optimal, and why not |
| `analysis.mdao` | OpenMDAO | `pip install "longeron[mdao]"` | continuous sizing, what-ifs, gradient-based optimization, external-tool binding |
| `analysis.smt` | Z3 | `pip install "longeron[smt]"` | is the requirement set consistent at all, which requirements conflict, exact feasibility bounds |

The interpreter remains the single source of semantics: every solver
result below is re-evaluated (or cross-checked) against the model itself.

In [ ]:
import sysml2
from sysml2.analysis import mdao, smt, trades

## A multi-mission UAV catalog

`examples/uav_missions.sysml` models one component catalog -- three
airframe families, tiered motors, props, batteries, and mission
equipment -- evaluated against **three mission contexts**, each a part
definition specializing the shared `MissionUAV` assembly with its own
equipment, metrics, and requirements:

* **`IsrUav`** -- loiter on station with a stabilized sensor (metric:
  `stationMinutes`);
* **`LogisticsUav`** -- fly a parcel out and return *empty*, the two legs
  costing different power (metric: `payloadRangeKgKm`);
* **`InterceptUav`** -- a one-way dash to catch a target crossing at 25
  m/s, first seen 3 km out (metric: `maxTargetSpeed`).

The airframes are genuinely different machines: a cheap rotor-only
**quad**; a **winged VTOL** that hovers on four props but cruises on its
2.6 m wing -- the two large props sit on *wingtip nacelles* where they
rotate against the tip vortices (modeled as a 1.28x bonus on the wing's
effective span efficiency, i.e. less induced drag), with two smaller lift
props atop the twin vertical stabilizers; and a rail-launched
**streamlined interceptor** (tiny CdA, single pusher motor, no hover
requirement -- and almost no payload).

All the physics lives in `calc def`s the interpreter evaluates directly --
momentum-theory hover power from disk loading, parasite + induced-drag
cruise power, drag-limited dash speed, and the lead-collision intercept
triangle -- so the model itself carries the analysis.

In [ ]:
model = sysml2.load("../examples/uav_missions.sysml")
missions = {
    "ISR": ("UavMissions::IsrUav", "stationMinutes"),
    "logistics": ("UavMissions::LogisticsUav", "payloadRangeKgKm"),
    "intercept": ("UavMissions::InterceptUav", "maxTargetSpeed"),
}
studies = {name: trades.TradeStudy(model, qname)
           for name, (qname, _) in missions.items()}
for name, study in studies.items():
    points = ", ".join(f"{p.name}[{len(p.variants)}]"
                       for p in study.points.values())
    print(f"{name:9s} -> {points}")

### The honest solver choice at this scale

CP-SAT's fixed-point integer arithmetic covers linear-ish catalogs
(`examples/drone_catalog.sysml` still demos `enumerate`/`explain` on it),
but `sqrt`, `pow`, conditionals, and calc invocations are beyond the
mapper -- and it says so instead of silently mis-encoding. With 243
candidates per mission there is nothing to prune anyway: 
`all_architectures()` walks the whole Cartesian space through the
interpreter, exactly, in well under a second -- and each infeasible mix
carries `violations`, the names of the constraints it breaks.

In [ ]:
try:
    studies["ISR"].enumerate()
except sysml2.analysis.AnalysisError as err:
    print(f"CP-SAT mapper: {err}\n")

spaces = {name: study.all_architectures()
          for name, study in studies.items()}
for name, archs in spaces.items():
    feasible = sum(a.verified for a in archs)
    print(f"{name:9s} {feasible:3d} of {len(archs)} mixes feasible")

## Mission 1: ISR -- the wing buys the loiter

On station, the quad must *hover* (momentum-theory power from disk
loading) while the winged families fly slow on wing lift; wing-borne
loiter costs less than a tenth of hover power, and the wingtip props'
induced-drag bonus stretches it further. What to look for: the front is a
real staircase now -- budget quads hold the cheap corner at 30-42
minutes, then the winged VTOL takes over and runs to nearly two hours.
The interceptor is absent entirely: its 0.9 kg bay cannot carry the
grade-2 sensor the mission requires.

In [ ]:
from sysml2.analysis import viz

isr_front = trades.pareto([a for a in spaces["ISR"] if a.verified],
                          minimize=("missionCost",),
                          maximize=("stationMinutes",))
isr_best = max(isr_front, key=lambda a: a.metrics["stationMinutes"])
isr_cheap = min(isr_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["ISR"], x="missionCost", y="stationMinutes",
    panel_y="missionMass", xlabel="mission cost (USD)",
    ylabel="time on station (min)", panel_ylabel="mission mass (kg)",
    annotate={"winged VTOL, big pack: 115 min": isr_best,
              "budget quad corner": isr_cheap},
    title="The winged VTOL owns endurance; quads keep the cheap corner")

## Mission 2: logistics -- out heavy, back empty

The delivery radius is what the battery sustains for the *asymmetric*
round trip -- outbound at parcel weight, return with the empty bay -- after
a fixed hover budget for takeoff, drop-off, and landing. Rotor-borne
cruise never escapes hover power, so the quad's radius stalls near the
8 km requirement while the winged VTOL turns the same packs into 35-85
kg-km of delivered payload-range. The interceptor cannot even load the
smallest parcel (`cargoFits`).

In [ ]:
log_front = trades.pareto([a for a in spaces["logistics"] if a.verified],
                          minimize=("missionCost",),
                          maximize=("payloadRangeKgKm",))
log_best = max(log_front, key=lambda a: a.metrics["payloadRangeKgKm"])
log_cheap = min(log_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["logistics"], x="missionCost", y="payloadRangeKgKm",
    panel_y="deliveryRadiusKm", xlabel="mission cost (USD)",
    ylabel="payload x radius (kg km)", panel_ylabel="radius (km)",
    annotate={"winged VTOL: 2.5 kg out to 34 km": log_best,
              "$716 quad: 1 kg to 9 km": log_cheap},
    title="Wings turn batteries into payload-range; quads just clear 8 km")

## Mission 3: intercept -- one-way dash, low drag wins

Dash speed is parasite-drag-limited (`(2 eta P / rho CdA)^(1/3)`), and the
reachable target speed inverts the lead-collision triangle at the
battery-limited dash duration. The interceptor's CdA is a fifth of the
quad's, and its single motor sips from packs that quad dash mixes
overload (`packPower`: four sprint motors out-draw every battery in the
catalog). What to look for: the interceptor family owns the entire upper
front; quads catch slow crossers cheaply; the winged VTOL dashes
respectably but pays its airframe premium.

In [ ]:
int_front = trades.pareto([a for a in spaces["intercept"] if a.verified],
                          minimize=("missionCost",),
                          maximize=("maxTargetSpeed",))
int_best = max(int_front, key=lambda a: a.metrics["maxTargetSpeed"])
int_cheap = min(int_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["intercept"], x="missionCost", y="maxTargetSpeed",
    panel_y="dashSpeed", xlabel="mission cost (USD)",
    ylabel="max catchable target speed (m/s)",
    panel_ylabel="dash speed (m/s)",
    annotate={"sprint dart: 50 m/s targets": int_best,
              "$773 quad catches 27 m/s": int_cheap},
    title="The streamlined dart owns the dash; sprint quads die on packPower")

### Why the dead mixes die

Every infeasible mix names the constraints it breaks -- the mix-level
answer CP-SAT's `explain()` gives at catalog level. The census per
mission is the design story in one table: sensors too heavy for small
bays, sprint motors out-drawing every pack, eco motors that cannot VTOL
the big battery, dashes that never catch the target.

In [ ]:
from collections import Counter

for name, archs in spaces.items():
    census = Counter(v for a in archs if not a.verified
                     for v in a.violations)
    print(f"{name:9s}", dict(census.most_common()))

## Across missions: is any one bird good at everything?

Project each mission's front onto the shared selection (airframe, motors,
props, battery). Three base mixes sit on *both* the ISR and logistics
fronts -- above all the winged VTOL with standard motors and slim props,
at either pack size: buy that bird and re-fit the payload bay between
sorties. Nothing reaches all three fronts; the interceptor's dash physics
really is a different aircraft.

In [ ]:
def base_mix(arch):
    keep = ("airframe", "motors", "props", "battery")
    return tuple(arch.selection[k] for k in keep)

fronts = {"ISR": isr_front, "logistics": log_front, "intercept": int_front}
membership = {}
for name, front in fronts.items():
    for arch in front:
        membership.setdefault(base_mix(arch), set()).add(name)
print(f"{'airframe':16s}{'motors':13s}{'props':12s}{'battery':10s} fronts")
for mix, names in sorted(membership.items(),
                         key=lambda kv: (-len(kv[1]), kv[0])):
    print("".join(f"{part:13s}" if i else f"{part:16s}"
                  for i, part in enumerate(mix))
          + "  " + ", ".join(sorted(names)))

### Brushing the mission space

`viz.parcoords` (the house anywidget: inline vanilla JS, brushes in a
narrow zone around each axis, editable intervals, `selected` synced back
to Python) now carries one line per *base mix*, scored on every mission
at once: each metric is the best that mix achieves over its equipment
options, 0 where no equipment choice is feasible. Dashed gray lines fail
every mission. Brush `stationMinutes` high and `maxTargetSpeed` high to
watch the space empty out -- no line survives both.

In [ ]:
cross_rows = []
cross_mixes = []
for arch in spaces["intercept"]:   # the 81 shared base mixes
    mix = dict(zip(("airframe", "motors", "props", "battery"),
                   base_mix(arch)))
    row = dict(mix)
    for name, metric in (("ISR", "stationMinutes"),
                         ("logistics", "payloadRangeKgKm"),
                         ("intercept", "maxTargetSpeed")):
        best = [a for a in spaces[name] if a.verified
                and base_mix(a) == base_mix(arch)]
        row[metric] = max((a.metrics[metric] for a in best), default=0.0)
    row["cost"] = arch.metrics["missionCost"]
    row["feasible"] = any(row[m] > 0 for m in
                          ("stationMinutes", "payloadRangeKgKm",
                           "maxTargetSpeed"))
    cross_rows.append(row)
    cross_mixes.append(arch)

pc = viz.parcoords(cross_rows, axes=[
    "airframe", "motors", "props", "battery", "cost",
    "stationMinutes", "payloadRangeKgKm", "maxTargetSpeed"])
pc

## Three missions, three shapes, one scale

`analysis.geometry` bakes each family parametrically from the selected
catalog values (stdlib math, ~1 ms, no CAD kernel): the quad's frame from
prop diameter + clearance, the winged VTOL's wing from span/chord/taper
with wingtip nacelles and stabilizer-top lift props, the interceptor as a
lathed slender body with a pusher prop. `geometry.lineup` merges the
endurance winner, the budget freighter corner, and the dash winner into
one to-scale scene -- the 2.6 m wing genuinely dwarfs the dart. Drag to
orbit, scroll to zoom, double-click to re-fit. (The viewer loads three.js
from a CDN, the one network dependency.)

In [ ]:
from sysml2.analysis import geometry, viewer3d

log_quad = min((a for a in spaces["logistics"] if a.verified
                and a.selection["airframe"] == "boxQuad"),
               key=lambda a: a.metrics["missionCost"])
scene = geometry.lineup(
    [geometry.mission_geometry(studies["ISR"], isr_best),
     geometry.mission_geometry(studies["logistics"], log_quad),
     geometry.mission_geometry(studies["intercept"], int_best)],
    labels=["ISR", "LOG", "INT"], gap=0.4)
viewer3d.mesh_viewer(
    scene, width_px=860,
    label=(f"ISR winner {isr_best.metrics['stationMinutes']:.0f} min | "
           f"budget freighter ${log_quad.metrics['missionCost']:.0f} | "
           f"dash winner {int_best.metrics['maxTargetSpeed']:.0f} m/s"))

### Linked selection: parallel coordinates -> 3D

Plain traitlets compose the widgets: observe `selected` on the parallel
coordinates and re-bake the first surviving base mix into the viewer --
brush the `airframe` axis through its three categories and watch the
shape switch family.

In [ ]:
import json

linked = viewer3d.mesh_viewer(
    geometry.mission_geometry(studies["intercept"], cross_mixes[0]),
    label=" / ".join(base_mix(cross_mixes[0])))

def show_first_selected(change):
    indices = json.loads(change["new"] or "[]")
    if indices:
        mix = cross_mixes[indices[0]]
        linked.mesh_json = json.dumps(
            geometry.mission_geometry(studies["intercept"], mix))
        linked.label = " / ".join(base_mix(mix))

pc.observe(show_first_selected, names="selected")
linked

## Continuous sizing: how fast should the ISR winner loiter?

Trades picked *which* components; `analysis.mdao` sizes what stays
continuous. `UavMissions::IsrPrime` freezes the ISR winner's masses and
efficiencies as a concrete part definition and leaves `loiterSpeed` free;
`build_problem` mirrors it onto an OpenMDAO `Problem` (attributes become
components evaluating through the interpreter, constraints and the
`IsrStation` requirement become `*_margin` outputs).

In [ ]:
build = mdao.build_problem(model, "UavMissions::IsrPrime",
                           requirements=("UavMissions::IsrStation",))
p = build.problem
p.run_model()
print("loiterPowerW:  ", round(float(p.get_val("loiterPowerW")[0]), 1))
print("stationMinutes:", round(float(p.get_val("stationMinutes")[0]), 1))
p.set_val("loiterSpeed", 21.0)   # what-if: loiter at transit speed
p.run_model()
print("at 21 m/s:     ", round(float(p.get_val("stationMinutes")[0]), 1),
      "min -- stationFloor margin",
      round(float(p.get_val("stationFloor_margin")[0]), 1))
p.set_val("loiterSpeed", 15.0)
p.run_model()

### The margin picture

Sweeping `loiterSpeed` shows the whole feasible band in one chart: the
90-minute station floor is comfortable at the slow end and crosses zero
just above 17 m/s -- that is the binding requirement; the stall and
transit-speed limits frame the band.

In [ ]:
fig = viz.margin_sweep_figure(
    p, "loiterSpeed", [11.0 + 0.26 * i for i in range(51)],
    build.constraints, xlabel="loiter speed (m/s)",
    title="Loitering above ~17 m/s breaks the 90-minute station floor")

Maximizing `stationMinutes` with SLSQP drives the loiter speed straight
into the stall bound -- by the first-order drag polar, the slower the
better:

In [ ]:
opt = mdao.build_problem(model, "UavMissions::IsrPrime", setup=False,
                         requirements=("UavMissions::IsrStation",))
mdao.add_optimization(opt, objective="stationMinutes",
                      design_vars={"loiterSpeed": (11.0, 24.0)},
                      maximize=True)
opt.problem.setup()
opt.problem.set_val("loiterSpeed", 16.0)
opt.problem.run_driver()
print(f"best loiter = {opt.problem.get_val('loiterSpeed')[0]:.2f} m/s "
      f"({opt.problem.get_val('stationMinutes')[0]:.0f} min on station)")

## Declared external analyses: swapping the aerodynamics fidelity

First-order physics belongs in the model as `calc def` bodies -- but
higher-fidelity tools live outside SysML. The convention shipped with the
example makes the *model* declare the binding:

```sysml
metadata def ExternalAnalysis { attribute component : String; }

calc def CruisePower {
    @ExternalAnalysis { component = "uav_aero:CruisePowerPolar"; }
    in massKg : Real;  in speed : Real;  ...
    return : Real = ...first-order drag polar...;
}
```

The calc's `in`/`return` parameters *are* the I/O contract.
`build_problem` validates them against the wrapped OpenMDAO component's
actual inputs/outputs (a mismatch fails with both name lists) and
`fidelity={"CruisePower": "external"}` swaps the interpreter-backed body
for the component -- here `examples/uav_aero.py`, a synthetic
Reynolds-and-stall-aware polar. Everything else in the Problem is
untouched, so lo-fi/hi-fi comparison is one keyword:

In [ ]:
import sys

if "../examples" not in sys.path:
    sys.path.insert(0, "../examples")   # the uav_aero entry point

lo = mdao.build_problem(model, "UavMissions::IsrPrime")
hi = mdao.build_problem(model, "UavMissions::IsrPrime",
                        fidelity={"CruisePower": "external"})
print("bound externals:", hi.externals)

speeds = [11.0 + 0.2 * i for i in range(51)]
station = {}
for name, b in (("first-order calc body", lo),
                ("uav_aero polar (external)", hi)):
    values = []
    for v in speeds:
        b.problem.set_val("loiterSpeed", v)
        b.problem.run_model()
        values.append(float(b.problem.get_val("stationMinutes")[0]))
    station[name] = values

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.0, 3.6), layout="constrained")
for (name, values), color in zip(station.items(), ("#2f6b8f", "#c2603e")):
    ax.plot(speeds, values, color=color, linewidth=1.6)
    best = max(range(len(speeds)), key=lambda i: values[i])
    ax.plot(speeds[best], values[best], "o", color=color, markersize=5)
    ax.annotate(f"{name}\nbest {values[best]:.0f} min "
                f"@ {speeds[best]:.1f} m/s",
                (speeds[best], values[best]), xytext=(10, -6),
                textcoords="offset points", fontsize=8, color=color)
ax.set_xlabel("loiter speed (m/s)")
ax.set_ylabel("time on station (min)")
ax.set_title("The Reynolds/stall-aware polar backs loiter off the stall "
             "and costs 40 min", fontsize=10, loc="left", color="#2b2d31")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#d9dbdf", linewidth=0.5)

The first-order body rewards flying ever slower; the external polar's
stall-adjacent drag rise pushes the optimum to ~12.5 m/s and takes the
promised endurance from 172 to 131 minutes. Same model, same declared
contract, one keyword -- that is what the annotation is for. (The
convention is a candidate for a shared library package once it has
earned its keep here.)

## Requirement consistency with Z3

`analysis.smt` answers questions over *unbounded reals*: is the
requirement set satisfiable at all, and what exactly bounds it? The
`IsrStation` requirement (90 minutes on station) is consistent with
`IsrPrime`'s physics -- and freeing `loiterSpeed` shows the *exact*
fastest loiter that still satisfies it:

In [ ]:
system = smt.to_smt(model, "UavMissions::IsrPrime",
                    requirements=("UavMissions::IsrStation",))
result = system.check()
print(result.status, {k: round(v, 1) for k, v in result.witness.items()
                      if k in ("loiterSpeed", "stationMinutes")})

freed = smt.to_smt(model, "UavMissions::IsrPrime",
                   requirements=("UavMissions::IsrStation",),
                   free=("loiterSpeed",))
bound, _ = freed.maximize("loiterSpeed")
print("fastest loiter satisfying the 90 min floor:", bound, "m/s")

Now demand 180 minutes -- more than the aircraft has at any legal speed --
via programmatic authoring (notebook 1), and ask *which* requirements
collide. The core names `DeepStare::longStation` against `aboveStall`
(only sub-stall speeds could stretch the battery that far) through the
defining equations, and correctly excludes the satisfiable 90-minute
floor:

In [ ]:
from sysml2 import model as M

deep = M.Definition(kind="requirement", name="DeepStare")
deep.add(M.Usage(kind="subject", name="uav", types=["IsrPrime"]))
deep.add(M.Usage(
    kind="constraint", name="longStation", constraint_kind="require",
    result=sysml2.parse_expression("uav.stationMinutes >= 180.0")))
model.find("UavMissions").add(deep)

conflicted = smt.to_smt(model, "UavMissions::IsrPrime",
                        requirements=("UavMissions::IsrStation",
                                      "UavMissions::DeepStare"),
                        free=("loiterSpeed",))
result = conflicted.check()
print(result.status)
for label in result.core:
    if not label.endswith(".value"):
        print(" ", label)

## Where this goes: the two-level loop

The pieces compose into classic mixed-discrete MDO: **the trade study
picks the architecture, OpenMDAO sizes it** -- `all_architectures()`
scores every mix exactly through the interpreter, the winner freezes into
a concrete sizing context whose continuous attributes SLSQP optimizes
(swapping declared external analyses in for the physics that outgrows
first-order calc bodies), while Z3 guards the requirement set before any
solver time is spent on an impossible ask.

### Real CAD when you need it

For actual CAD output -- STEP for a printable frame -- `geometry.
to_cadquery` rebuilds the quad parametrically as cadquery solids behind
the `cad` extra (the OCC kernel is ~1 GB, which is why the mesh pipeline
above never touches it). This cell degrades to a note when cadquery is
not installed:

In [ ]:
try:
    params = geometry.mission_params(studies["logistics"], log_quad)
    assembly = geometry.to_cadquery(
        prop_diameter_in=params["prop_diameter"] / geometry.IN,
        motor_mass=params["motor_mass"],
        battery_mass=params["battery_mass"], esc_mass=0.014)
    print(f"cadquery assembly with {len(assembly.children)} parts -- "
          "assembly.export('drone.step') exports STEP")
except ImportError:
    print('optional: pip install "longeron[cad]" enables STEP export '
          "-- skipping here")